In [7]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
data = load_wine(as_frame=True)
df = data.frame.copy()

# Convert multiclass target into binary target
df["target"] = (df["target"] == 0).astype(int)

In [ ]:
selected_features = [
    "alcohol",
    "flavanoids",
    "color_intensity",
    "proline"
]

df = df[selected_features + ["target"]]

print("First 5 rows of the dataset:")
print(df.head())
print("\nSelected features:", selected_features)
print("\nDataset shape:", df.shape)
print("\nTarget value counts:")
print(df["target"].value_counts())

First 5 rows of the dataset:
   alcohol  flavanoids  color_intensity  proline  target
0    14.23        3.06             5.64   1065.0       1
1    13.20        2.76             4.38   1050.0       1
2    13.16        3.24             5.68   1185.0       1
3    14.37        3.49             7.80   1480.0       1
4    13.24        2.69             4.32    735.0       1

Selected features: ['alcohol', 'flavanoids', 'color_intensity', 'proline']

Dataset shape: (178, 5)

Target value counts:
target
0    119
1     59
Name: count, dtype: int64


In [9]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier(random_state=42))])

In [11]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nModel Accuracy:", round(accuracy, 4))
print("\nConfusion Matrix:")
print(cm)


Model Accuracy: 0.9444

Confusion Matrix:
[[23  1]
 [ 1 11]]


In [13]:
print("\nEnter values for a new prediction.")
print("Press Enter to use the mean value for that feature.\n")

default_values = X_train.mean(numeric_only=True)
user_values = []

for col in X.columns:
    value = input(f"{col} [{default_values[col]:.4f}]: ").strip()
    if value == "":
        user_values.append(default_values[col])
    else:
        user_values.append(float(value))

user_df = pd.DataFrame([user_values], columns=X.columns)

prediction = model.predict(user_df)[0]
probabilities = model.predict_proba(user_df)[0]

print("\nPrediction Result:")
if prediction == 1:
    print("Predicted class: class_0")
else:
    print("Predicted class: not_class_0")

print("\nPrediction Probabilities:")
print(f"not_class_0: {probabilities[0]:.4f}")
print(f"class_0:     {probabilities[1]:.4f}")


Enter values for a new prediction.
Press Enter to use the mean value for that feature.



alcohol [12.9546]:  14.2
flavanoids [2.0864]:  3.2
color_intensity [5.0342]:  7.3
proline [740.4577]:  500



Prediction Result:
Predicted class: not_class_0

Prediction Probabilities:
not_class_0: 0.6800
class_0:     0.3200
